# Pooled multi-dataset PFN (Med3D → TabPFN)

Train a single model on the UNION of datasets (e.g., `gist` + `lipo`) to test cross-cancer generalization.


In [1]:
# Use conda env packages (avoid user site-packages)
import os, sys, site
os.environ['PYTHONNOUSERSITE']='1'
usr=site.getusersitepackages(); sys.path=[p for p in sys.path if p!=usr]
import numpy as np, sklearn
print('numpy:', np.__version__, '|', np.__file__)
print('sklearn:', sklearn.__version__, '|', sklearn.__file__)


numpy: 2.2.6 | c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\numpy\__init__.py
sklearn: 1.7.2 | c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\sklearn\__init__.py


In [2]:
# Imports & setup
from pathlib import Path
import sys, json, datetime
import pandas as pd

def find_project_root():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand/'SAM-Med3D-main').is_dir() and (cand/'notebooks').is_dir():
            return cand
    return Path.cwd().resolve()

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT:', PROJECT_ROOT)

import yaml
from med3pipe import (
    find_default_sam3d_root, prepare_for_sam3d, split_validation,
    build_sam3d_model, default_feature_dirs, extract_embeddings_train_val,
    load_roi_features, load_labels_from_sheet, build_y,
    standardize_pca, train_eval_tabpfn
)
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score


PROJECT_ROOT: C:\Users\cahel\Desktop\Med3Tab-PFN


c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Config
config_path = PROJECT_ROOT/'configs'/'datasets.yaml'
cfg = yaml.safe_load(open(config_path,'r',encoding='utf-8'))
datasets_cfg = cfg.get('datasets', {})
# Select which datasets to include in the pool (None = all)
SELECTED = None  # e.g., ['gist','lipo']
dataset_names = [k for k in datasets_cfg.keys() if (SELECTED is None or k in SELECTED)]
print('Pooling datasets:', dataset_names)

def resolve_dataset_root(entry: dict, project_root: Path) -> Path:
    ds_root = Path(entry.get('dataset_root', entry.get('category','')))
    if ds_root.is_absolute() and ds_root.exists(): return ds_root
    cand = project_root/ds_root
    if cand.exists(): return cand
    cat = entry.get('category')
    if cat:
        cand2 = project_root/cat
        if cand2.exists(): return cand2
        cand3 = project_root/'data'/cat
        if cand3.exists(): return cand3
    return cand

USE_GPU = None  # 'cuda' | 'cpu' | None (auto)
MODEL_TYPE = 'vit_b_ori'
IMG_SIZE = 128


Pooling datasets: ['gist', 'lipo']


In [4]:
# Prepare, extract, and pool
sam3d_root = find_default_sam3d_root()
ckpt = sam3d_root/'ckpt'/'sam_med3d_turbo.pth'
checkpoint = ckpt if ckpt.exists() else None

paths_by = {}
specs = {}  # per-dataset config snapshot

# 1) Prepare and split each dataset
for name in dataset_names:
    e = datasets_cfg[name]
    ds_root = resolve_dataset_root(e, PROJECT_ROOT)
    cat, ct = e['category'], e['ct_name']
    prep = e.get('prepare', {})
    split = e.get('split', {})
    prepared, paths = prepare_for_sam3d(
        dataset_root=ds_root, sam3d_root=sam3d_root,
        category=cat, ct_name=ct,
        case_glob=prep.get('case_glob'), max_cases=prep.get('max_cases'),
)
    split_validation(paths, split_ratio=float(split.get('ratio',0.8)), seed=int(split.get('seed',2025)), copy=True)
    paths_by[name] = paths
    specs[name] = {
        'root': str(ds_root), 'category': cat, 'ct_name': ct,
        'labels': e.get('labels', {}), 'img_size': int(e.get('extraction',{}).get('img_size', IMG_SIZE))
    }

# 2) Build model once
model = build_sam3d_model(sam3d_root=sam3d_root, model_type=MODEL_TYPE, checkpoint=checkpoint, device=USE_GPU, eval_mode=True)

# 3) Extract embeddings for all datasets
feat_by = {}
for name in dataset_names:
    cat = specs[name]['category']; ct = specs[name]['ct_name']
    img_size = specs[name]['img_size']
    feat_dirs = default_feature_dirs(sam3d_root, category=cat, ct_name=ct)
    extract_embeddings_train_val(paths_by[name], model, sam3d_root=sam3d_root, img_size=img_size, feature_dirs=feat_dirs, device=USE_GPU)
    feat_by[name] = feat_dirs

# 4) Load ROI features + labels per dataset (filter to labeled cases so X and y lengths match)
Xtr_list, ytr_list = [], []
Xva_list, yva_list = [], []
ids_val_all, dset_val_all = [], []
for name in dataset_names:
    paths = paths_by[name]; feat = feat_by[name]
    labs = specs[name]['labels']
    ds_root = Path(specs[name]['root'])
    sheet_csv = labs.get('sheet_csv','sheet.csv')
    sheet_csv = (ds_root/sheet_csv) if not Path(sheet_csv).is_absolute() else Path(sheet_csv)

    ds_col = labs.get('dataset_col', 'Dataset')
    ds_name = labs.get('dataset_name', None)
    try:
        df, lab_map = load_labels_from_sheet(
            sheet_csv=sheet_csv, dataset_col=ds_col, dataset_name=ds_name,
            subject_col=labs.get('subject_col','Subject'),
            label_col=labs.get('label_col','Diagnosis_binary'),
            case_suffix=labs.get('case_suffix','_CT'),
        )
    except ValueError as e:
        print(f'[WARN] {name}: {e}. Falling back to no dataset filter.')
        df, lab_map = load_labels_from_sheet(
            sheet_csv=sheet_csv, dataset_name=None,
            subject_col=labs.get('subject_col','Subject'),
            label_col=labs.get('label_col','Diagnosis_binary'),
            case_suffix=labs.get('case_suffix','_CT'),
        )

    # Train features/ids
    Xtr, idtr = load_roi_features(feat.train_dir, paths.labels_tr)
    if Xtr.size:
        idtr_arr = np.array(idtr)
        # Mask for which ids have labels
        def _clean(c):
            return c[:-4] if c.endswith('.nii') else c
        mask_tr = np.array([_clean(c) in lab_map for c in idtr_arr], dtype=bool)
        if mask_tr.any():
            Xtr = Xtr[mask_tr]
            ytr = np.array([lab_map[_clean(c)] for c in idtr_arr[mask_tr]], dtype=int)
            Xtr_list.append(Xtr); ytr_list.append(ytr)
        else:
            print(f'[WARN] {name}: no labeled TRAIN cases matched; skipping from train pool.')
    else:
        print(f'[WARN] {name}: no TRAIN embeddings found.')

    # Val features/ids
    Xva, idva = load_roi_features(feat.val_dir, paths.labels_val)
    if Xva.size:
        idva_arr = np.array(idva)
        mask_va = np.array([_clean(c) in lab_map for c in idva_arr], dtype=bool)
        if mask_va.any():
            Xva_f = Xva[mask_va]
            yva_f = np.array([lab_map[_clean(c)] for c in idva_arr[mask_va]], dtype=int)
            Xva_list.append(Xva_f); yva_list.append(yva_f)
            ids_val_all += idva_arr[mask_va].tolist()
            dset_val_all += [name]*int(mask_va.sum())
        else:
            print(f'[WARN] {name}: no labeled VAL cases matched; skipping from val pool.')
    else:
        print(f'[WARN] {name}: no VAL embeddings found.')

# 5) Pool across datasets
if not Xtr_list or not Xva_list:
    raise RuntimeError('No labeled features found to pool. Check your sheets and case suffixes.')
X_train = np.vstack(Xtr_list); y_train = np.concatenate(ytr_list)
X_val   = np.vstack(Xva_list); y_val   = np.concatenate(yva_list)
print('Pooled shapes -> X_train:', X_train.shape, 'y_train:', y_train.shape, '| X_val:', X_val.shape, 'y_val:', y_val.shape)

# 6) Standardize + PCA (fit on pooled TRAIN)
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
out_dir = (PROJECT_ROOT/'notebooks'/'tabpfn_runs'/('pooled_' + '-'.join(dataset_names) + '_' + timestamp)).resolve()
preproc_dir = out_dir/'preproc'
Xtr_p, Xva_p, scaler, pca = standardize_pca(X_train, X_val, n_components_max=500, random_state=42, save_dir=preproc_dir)

# 7) Train TabPFN on pooled TRAIN and evaluate on pooled VAL
res = train_eval_tabpfn(X_train=Xtr_p, y_train=y_train, X_val=Xva_p, y_val=y_val, ids_val=ids_val_all, out_dir=out_dir)
print('Overall pooled metrics:', res['metrics'])

# 8) Per-dataset breakdown using saved predictions
pred = pd.read_csv(res['pred_path'])
map_df = pd.DataFrame({'case_id': ids_val_all, 'dataset': dset_val_all})
pred = pred.merge(map_df, on='case_id', how='left')
rows=[]
for dname, g in pred.groupby('dataset'):
    y_true = g['true_label'].to_numpy(); y_pred = g['pred_label'].to_numpy()
    acc = float(accuracy_score(y_true, y_pred))
    f1  = float(f1_score(y_true, y_pred, average='macro'))
    auc = None
    if 'proba_1' in g.columns:
        try:
            auc = float(roc_auc_score(y_true, g['proba_1'].to_numpy()))
        except Exception:
            auc = None
    rows.append({'dataset': dname, 'accuracy': acc, 'macro_f1': f1, 'roc_auc': auc, 'n': int(len(g))})
breakdown = pd.DataFrame(rows).sort_values('dataset')
display(breakdown)
sum_path = PROJECT_ROOT/'notebooks'/'multi_pooled_summary.csv'
breakdown.to_csv(sum_path, index=False)
print('Saved per-dataset summary to:', sum_path)


Prepared 25 cases ...
Prepared 50 cases ...
Prepared 75 cases ...
Prepared 100 cases ...
Prepared 125 cases ...
Prepared 150 cases ...
Prepared 175 cases ...
Prepared 200 cases ...
Prepared 225 cases ...
Done. Prepared 246 cases to C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST
Validation set copied to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\validation\gist\ct_GIST | Train: 196 | Val: 50
Prepared 25 cases ...
Prepared 50 cases ...
Prepared 75 cases ...
Prepared 100 cases ...
Done. Prepared 115 cases to C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\lipo\ct_LIPO
Validation set copied to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\validation\lipo\ct_LIPO | Train: 92 | Val: 23
To extract: 246 from C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST\imagesTr
Done extracting to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main

c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\tabpfn\classifier.py:465: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  check_cpu_warning(


Overall pooled metrics: {'accuracy': 1.0, 'macro_f1': 1.0, 'roc_auc': 1.0, 'confusion_matrix': [[1, 0, 0], [0, 29, 0], [0, 0, 43]]}


,dataset,accuracy,macro_f1,roc_auc,n
0,gist,1.0,1.0,None,50
1,lipo,1.0,1.0,None,23


Saved per-dataset summary to: C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\multi_pooled_summary.csv
